# 040 — Árboles de decisión y reglas interpretables

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Árbol de decisión (CART):** partición recursiva del espacio con preguntas `xⱼ ≤ t`;
cada hoja predice la mayoría (o media). Cada camino raíz→hoja es una regla legible.

**Elección del split — impureza:**

```text
Gini: G = 1 − Σ pₖ²        Entropía: H = −Σ pₖ log₂ pₖ
Ganancia: Δ = I(padre) − (n_L/n)·I(L) − (n_R/n)·I(R)
```

El algoritmo es voraz: elige la Δ máxima nodo a nodo (no garantiza el árbol óptimo global).

**Sobreajuste y poda:** un árbol sin límites memoriza el train (alta varianza).
Pre-poda: `max_depth`, `min_samples_leaf`. Post-poda costo-complejidad:
minimizar `R(T) + α·|hojas|` con α elegido por validación cruzada.

**Interpretabilidad condicionada:** vale en árboles pequeños; la importancia por impureza
está sesgada (alta cardinalidad, correlación) y describe al modelo, no al fenómeno.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Gini(padre) = 1 − (0.5²+0.5²) = 0.5.
Split A: cada hijo tiene Gini = 1 − (0.75²+0.25²) = 1 − 0.625 = 0.375; ponderado =
(4/8)·0.375 + (4/8)·0.375 = 0.375; **Δ_A = 0.125**.
Split B: hijo (4+,2−): 1 − ((2/3)²+(1/3)²) = 1 − 5/9 ≈ 0.444; hijo (0+,2−): 0;
ponderado = (6/8)·0.444 + (2/8)·0 = 0.333; **Δ_B ≈ 0.167**. CART elige **B**: crear una
hoja pura pequeña reduce más impureza que dos hijos moderadamente mejores.

**Ejercicio 2.** H(padre) = 1 bit. Split A: H(hijo) = −0.75·log₂0.75 − 0.25·log₂0.25 ≈
0.811; ponderado 0.811 → Δ_A ≈ 0.189. Split B: H(4,2) ≈ 0.918, H(0,2) = 0; ponderado =
0.75·0.918 ≈ 0.689 → Δ_B ≈ 0.311. **El ganador no cambia** (B). Gini se prefiere por
costo: evita logaritmos y en la práctica produce árboles casi idénticos.

**Ejercicio 3.** (a) La brecha 1.00→0.71 es varianza pura: memorización. La brecha
0.88→0.84 es pequeña: el árbol podado generaliza. (b) El podado: 0.84 > 0.71 en el único
dato que importa (fuera de train). (c) Verificar que α se eligió con validación cruzada y
no mirando el test, y medir una única vez sobre test para la cifra final.

**Ejercicio 4.** Con el umbral t y accuracy a del laboratorio, la regla es
`SI x ≥ t ENTONCES positivo`. La regla contraria acierta exactamente los casos que la
original falla, así que su accuracy es 1 − a: en clasificación binaria un clasificador
peor que el azar se convierte en uno mejor invirtiendo las etiquetas — por eso el peor
caso real es 0.5, no 0.


In [ ]:
result = run_lab("ml", seed=40)
assert result["kind"] == "ml"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicios 1 y 2 — Gini y entropía verificados
import math

def gini(pos, neg):
    n = pos + neg
    if n == 0:
        return 0.0
    p, q = pos / n, neg / n
    return 1 - (p * p + q * q)

def entropia(pos, neg):
    n = pos + neg
    out = 0.0
    for c in (pos, neg):
        if c:
            p = c / n
            out -= p * math.log2(p)
    return out

def ganancia(criterio, padre, hijos):
    n = sum(sum(h) for h in hijos)
    return criterio(*padre) - sum((sum(h) / n) * criterio(*h) for h in hijos)

for nombre, crit in (("Gini", gini), ("Entropía", entropia)):
    d_a = ganancia(crit, (4, 4), [(3, 1), (1, 3)])
    d_b = ganancia(crit, (4, 4), [(4, 2), (0, 2)])
    print(f"{nombre}: Δ_A={d_a:.3f}  Δ_B={d_b:.3f}  → gana {'B' if d_b > d_a else 'A'}")


In [ ]:
# Ejercicio 4 — el laboratorio como decision stump
result = run_lab("ml", seed=40)
sel = result["result"]["selected"]
t, a = sel["threshold"], sel["accuracy"]
print(f"Regla: SI x >= {t} ENTONCES positivo   (accuracy desarrollo = {a:.2f})")
print(f"Regla invertida: accuracy = {1 - a:.2f}")
# En binario, invertir las predicciones convierte accuracy a en 1−a:
# el peor clasificador útil es el azar (0.5), no el 0.


## Reflexión

1. El laboratorio elige un umbral único sobre una sola feature: es exactamente un árbol de
   profundidad 1 (*decision stump*). ¿Qué ganancia Gini corresponde al umbral que
   selecciona y qué limitación compartida tienen el stump y el barrido del laboratorio?
2. Si dos features están perfectamente correlacionadas, ¿qué pasa con la "importancia" que
   el árbol asigna a cada una entre distintas semillas, y por qué eso invalida leerla como
   relevancia del fenómeno?
3. ¿En qué caso concreto la pre-poda (parar temprano) descartaría un split que la
   post-poda conservaría? Da un ejemplo con dos features binarias.
